<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/notebooks/06_channel_dilution_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Channel-dilution mechanism audit

**What this notebook tests:** notebook 05 found that the polarity-by-size interaction is strong on ResNet18 but weak/non-significant on EfficientNet-B0. This notebook tests one specific, pre-specified explanation: EfficientNet-B0's target layer has 2.5x more channels (1280 vs. 512) than ResNet18's, so Grad-CAM's channel-averaged weighting may dilute a narrow polarity-carrying signal more on EfficientNet-B0.

**The test:** recompute EfficientNet-B0's Grad-CAM using only its top-512 gradient-weighted channels (matching ResNet18's channel count) instead of all 1280, and check whether the interaction strengthens. This is a single, pre-specified prediction — not a search for a specification that produces significance — so the result is reported honestly whichever way it comes out.

**Result of running this once already:** the interaction strengthened (coef -0.028, p=0.011 → coef -0.048, p<0.001), supporting channel-count dilution as a real contributor to the architecture-dependence. This notebook is the clean, documented version of that test for reproducibility.

## Path reference — what each file is and where it comes from

| Variable | What it is | Produced by |
|---|---|---|
| `base` | Root folder of the CASIA v2 image set (`Tp`, `Au`, list files) | Your own Drive upload |
| `tp_dir` | Folder of tampered (copy-move + spliced) images | Subfolder of `base` |
| `gt_dir` | Folder of ground-truth tampering masks | Your own Drive upload (Pham et al. groundtruth) |
| `casia_effnet_best.pt` | Trained EfficientNet-B0 classifier weights | Notebook 03 |
| `splice_features.csv` | Per-image splice_size_frac, polarity, abs_contrast | Notebook 02 |
| `gradcam_iou_effnet.csv` | Standard (all-1280-channel) Grad-CAM IoU, EfficientNet-B0 | Notebook 04 |
| `gradcam_iou_effnet_top512.csv` | **New in this notebook** — top-512-channel-restricted Grad-CAM IoU | This notebook, saved at the end |

Adjust `base` and `gt_dir` below to match your own Drive layout — they should be identical to the paths used in notebooks 01–04.

## Setup — mount Drive, rebuild file lists, load EfficientNet-B0

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import statsmodels.formula.api as smf
from PIL import Image

# --- Same paths as notebooks 01-04. Adjust if your Drive layout differs. ---
base = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised"
tp_dir = os.path.join(base, "Tp")
gt_dir = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_Groundtruth/CASIA2.0_Groundtruth"

with open(os.path.join(base, "tp_list.txt")) as f:
    tp_list_content = [line.strip() for line in f if line.strip()]
tp_files_final = sorted(set(tp_list_content) & set(os.listdir(tp_dir)))
spliced_files = [f for f in tp_files_final if f.split("_")[1] == "D"]
print(f"Spliced-only file count: {len(spliced_files)}")  # expect 1828

def get_mask_path(image_filename):
    stem = os.path.splitext(image_filename)[0]
    return os.path.join(gt_dir, f"{stem}_gt.png")

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load the same trained EfficientNet-B0 checkpoint used in notebooks 03-05 ---
effnet = models.efficientnet_b0(weights=None)
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_effnet_best.pt", map_location=device))
effnet = effnet.to(device)
effnet.eval()
print("EfficientNet-B0 checkpoint loaded.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spliced-only file count: 1828
EfficientNet-B0 checkpoint loaded.


In [8]:
effnet = models.efficientnet_b0(weights=None)
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_effnet_best.pt", map_location=device))
effnet = effnet.to(device)
effnet.eval()
print("EfficientNet-B0 checkpoint loaded.")

# Also load ResNet18 briefly, just to verify the channel-count premise this notebook tests
resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_resnet_best.pt", map_location=device))
resnet = resnet.to(device)
resnet.eval()

print(f"\nResNet18 target layer channels: {resnet.layer4[-1].conv2.out_channels}")
print(f"EfficientNet-B0 target layer channels: {effnet.features[-1][0].out_channels}")

EfficientNet-B0 checkpoint loaded.

ResNet18 target layer channels: 512
EfficientNet-B0 target layer channels: 1280


## Top-K restricted Grad-CAM
Same mechanism as standard Grad-CAM (forward/backward hooks, channel-wise gradient averaging), except only the `top_k` channels with the largest |weight| contribute — all other channels are zeroed out before the weighted sum. Setting `top_k=512` matches ResNet18's channel count exactly.

In [5]:
class GradCAM_TopK:
    def __init__(self, model, target_layer, top_k=None):
        self.model = model
        self.gradients = None
        self.activations = None
        self.top_k = top_k
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[0, target_class]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)

        if self.top_k is not None:
            flat_weights = weights.squeeze().abs()
            k = min(self.top_k, flat_weights.shape[0])
            topk_idx = torch.topk(flat_weights, k).indices
            mask = torch.zeros_like(flat_weights)
            mask[topk_idx] = 1.0
            weights = weights * mask.view(1, -1, 1, 1)

        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam

gradcam_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def compute_iou(cam, gt_mask_binary):
    cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize(
        (gt_mask_binary.shape[1], gt_mask_binary.shape[0]), Image.BILINEAR
    )) / 255.0
    thresh = cam_resized.mean()
    cam_binary = cam_resized > thresh
    intersection = np.logical_and(cam_binary, gt_mask_binary).sum()
    union = np.logical_or(cam_binary, gt_mask_binary).sum()
    if union == 0:
        return None
    return intersection / union

## Run top-512 restricted Grad-CAM + IoU on all 1,828 spliced images

In [6]:
gradcam_effnet_topk = GradCAM_TopK(effnet, effnet.features[-1], top_k=512)

records_topk = []
for i, fname in enumerate(spliced_files):
    img_path = os.path.join(tp_dir, fname)
    mask_path = get_mask_path(fname)
    if not os.path.exists(mask_path):
        continue

    img_pil = Image.open(img_path).convert("RGB")
    input_tensor = gradcam_transform(img_pil).unsqueeze(0).to(device)
    cam = gradcam_effnet_topk.generate(input_tensor, target_class=1)

    orig_size = img_pil.size
    mask_pil = Image.open(mask_path).convert("L").resize(orig_size, Image.NEAREST)
    gt_mask_binary = np.array(mask_pil) > 127

    iou = compute_iou(cam, gt_mask_binary)
    if iou is None:
        continue

    records_topk.append({"filename": fname, "iou": iou})
    if (i + 1) % 200 == 0:
        print(f"Processed {i+1}/{len(spliced_files)}")

iou_df_topk = pd.DataFrame(records_topk)
iou_df_topk.to_csv("/content/drive/MyDrive/CASIA2.0/gradcam_iou_effnet_top512.csv", index=False)
print(f"\nDone. {len(iou_df_topk)} of {len(spliced_files)} images.")
print(iou_df_topk.describe())

Processed 200/1828
Processed 400/1828
Processed 600/1828
Processed 800/1828
Processed 1000/1828
Processed 1200/1828
Processed 1400/1828
Processed 1600/1828
Processed 1800/1828

Done. 1828 of 1828 images.
               iou
count  1828.000000
mean      0.163725
std       0.164773
min       0.000000
25%       0.033126
50%       0.107635
75%       0.253087
max       0.872692


## Compare: standard (1280-channel) vs. top-512-restricted regression
Same median-split moderation model as notebook 05, run on the top-512 IoU values, for direct comparison against the standard EfficientNet-B0 result already obtained.

In [7]:
features_df = pd.read_csv("/content/drive/MyDrive/CASIA2.0/splice_features.csv")

merged_topk = features_df.merge(iou_df_topk, on="filename", how="inner")
merged_topk["polarity_bin"] = (merged_topk["polarity"] == "dark_on_bright").astype(int)
median_size_topk = merged_topk["splice_size_frac"].median()
merged_topk["large_splice_median"] = (merged_topk["splice_size_frac"] >= median_size_topk).astype(int)

model_topk = "iou ~ polarity_bin * large_splice_median + abs_contrast"
ols_topk = smf.ols(formula=model_topk, data=merged_topk).fit(cov_type="HC3")
print("=== EfficientNet-B0, top-512 channels, median split ===")
print(ols_topk.summary().tables[1])

print("\nCompare against the standard (1280-channel) result from notebook 05:")
print("polarity_bin:large_splice_median = -0.0279, p=0.011 (standard, full channels)")

=== EfficientNet-B0, top-512 channels, median split ===
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            0.0457      0.005     10.112      0.000       0.037       0.055
polarity_bin                         0.0003      0.003      0.086      0.931      -0.006       0.007
large_splice_median                  0.2387      0.008     29.218      0.000       0.223       0.255
polarity_bin:large_splice_median    -0.0476      0.011     -4.232      0.000      -0.070      -0.026
abs_contrast                         0.0002   8.33e-05      2.297      0.022    2.81e-05       0.000

Compare against the standard (1280-channel) result from notebook 05:
polarity_bin:large_splice_median = -0.0279, p=0.011 (standard, full channels)


## Summary

| | Standard (1280 channels) | Top-512 restricted |
|---|---|---|
| `polarity_bin:large_splice_median` coef | -0.0279 | (see output above) |
| p-value | 0.011 | (see output above) |

If the top-512 result shows a larger-magnitude, more significant coefficient than the standard result, this supports channel-count dilution as a real contributor to the ResNet18/EfficientNet-B0 discrepancy — report this as a tested, confirmed mechanism in the Discussion section, not merely a candidate explanation. State clearly in the paper that this is a *modified* Grad-CAM variant used diagnostically, not a claim that EfficientNet-B0's standard Grad-CAM shows the effect.